# House Price Prediction

## Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

## Load the data

In [1]:
df = pd.read_csv("data.csv")
df.head()

                  date      price  bedrooms  bathrooms  sqft_living  sqft_lot  floors  waterfront  view  condition  sqft_above  sqft_basement  yr_built  yr_renovated                    street       city  statezip country
0  2014-05-02 00:00:00   313000.0       3.0       1.50         1340      7912     1.5           0     0          3        1340              0      1955          2005      18810 Densmore Ave N  Shoreline  WA 98133     USA
1  2014-05-02 00:00:00  2384000.0       5.0       2.50         3650      9050     2.0           0     4          5        3370            280      1921             0           709 W Blaine St    Seattle  WA 98119     USA
2  2014-05-02 00:00:00   342000.0       3.0       2.00         1930     11947     1.0           0     0          4        1930              0      1966             0  26206-26214 143rd Ave SE       Kent  WA 98042     USA
3  2014-05-02 00:00:00   420000.0       3.0       2.25         2000      8030     1.0           0     0          4  

In [1]:
df.shape

(4600, 18)

In [1]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [1]:
df.isnull().sum()

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

## Clean up the target column

In [1]:
print("Rows with price == 0:", (df['price'] == 0).sum())
df = df[df['price'] != 0]
print("Remaining rows:", df.shape[0])

Rows with price == 0: 49
Remaining rows: 4551

In [1]:
df['sale_year'] = pd.to_datetime(df['date']).dt.year
df['house_age'] = df['sale_year'] - df['yr_built']
df[['date', 'sale_year', 'yr_built', 'house_age']].head()

                  date  sale_year  yr_built  house_age
0  2014-05-02 00:00:00       2014      1955         59
1  2014-05-02 00:00:00       2014      1921         93
2  2014-05-02 00:00:00       2014      1966         48
3  2014-05-02 00:00:00       2014      1963         51
4  2014-05-02 00:00:00       2014      1976         38

## Remove price outliers (IQR method)

In [1]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

before = df.shape[0]
df = df[(df["price"] >= lower) & (df["price"] <= upper)]
print(f"Rows before: {before}, after: {df.shape[0]}")

Rows before: 4551, after: 4311

## New features


In [1]:
df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df[['is_renovated', 'total_rooms']].head()

   is_renovated  total_rooms
0             1         4.50
2             0         5.00
3             0         5.25
4             1         6.50
5             1         3.00

# Data Preprocessing

In [1]:
X = df.drop(columns=['price', 'date', 'country', 'street'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (3448, 18)  Test: (863, 18)

## Encode `city` and `statezip`

In [1]:
zip_price_map = df.loc[X_train.index].groupby('statezip')['price'].median()
city_price_map = df.loc[X_train.index].groupby('city')['price'].median()
global_median = y_train.median()

X_train = X_train.copy()
X_test = X_test.copy()

X_train['zip_price_median'] = df.loc[X_train.index, 'statezip'].map(zip_price_map)
X_test['zip_price_median'] = df.loc[X_test.index, 'statezip'].map(zip_price_map).fillna(global_median)

X_train['city_price_median'] = df.loc[X_train.index, 'city'].map(city_price_map)
X_test['city_price_median'] = df.loc[X_test.index, 'city'].map(city_price_map).fillna(global_median)

X_train = X_train.drop(columns=['statezip', 'city'])
X_test = X_test.drop(columns=['statezip', 'city'])

print("Final feature count:", X_train.shape[1])
print(list(X_train.columns))

Final feature count: 18
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'sale_year', 'house_age', 'is_renovated', 'total_rooms', 'zip_price_median', 'city_price_median']

# Linear Regression

In [1]:
model_LR = LinearRegression()
model_LR.fit(X_train, y_train)
y_pred_lr = model_LR.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Mean Absolute Error: ", mae_lr)
print("Mean Squared Error: ", mse_lr)
print("Root Mean Squared Error: ", rmse_lr)
print("R2-score: ", r2_lr)

Mean Absolute Error:  79446.1062253773
Mean Squared Error:  12291616091.846779
Root Mean Squared Error:  110867.56104400773
R2-score:  0.7377947017415412

# Support Vector Regression

In [1]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

model_svm = SVR(kernel='rbf')
model_svm.fit(X_train_scaled, y_train_scaled)

y_pred_svr = y_scaler.inverse_transform(
    model_svm.predict(X_test_scaled).reshape(-1, 1)
).ravel()

mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("Mean Absolute Error: ", mae_svr)
print("Mean Squared Error: ", mse_svr)
print("Root Mean Squared Error: ", rmse_svr)
print("R2-score: ", r2_svr)

Mean Absolute Error:  75314.78778690858
Mean Squared Error:  12078476795.302282
Root Mean Squared Error:  109902.12370697065
R2-score:  0.7423413986448164

# Decision Tree

In [1]:
model_dt = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Mean Absolute Error:", mae_dt)
print("Mean Squared Error:", mse_dt)
print("Root Mean Squared Error:", rmse_dt)
print("R2-score:", r2_dt)

Mean Absolute Error: 84261.6745494452
Mean Squared Error: 15025921270.173763
Root Mean Squared Error: 122580.26460313163
R2-score: 0.6794663827104547

# Random Forest

In [1]:
params = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)
rf_random = RandomizedSearchCV(
    estimator=rf, param_distributions=params,
    n_iter=10, cv=3, random_state=42, n_jobs=-1
)
rf_random.fit(X_train, y_train)
print("Best params:", rf_random.best_params_)

Best params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 20}

In [1]:
y_pred_rf = rf_random.best_estimator_.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Mean Absolute Error:", mae_rf)
print("Mean Squared Error:", mse_rf)
print("Root Mean Squared Error:", rmse_rf)
print("R2-score:", r2_rf)

Mean Absolute Error: 72331.12178259183
Mean Squared Error: 11400172829.442041
Root Mean Squared Error: 106771.59186526181
R2-score: 0.7568110088530504

# Gradient Boosting (new)


In [1]:
model_hgb = HistGradientBoostingRegressor(
    random_state=42, max_iter=500, learning_rate=0.03, max_depth=6
)
model_hgb.fit(X_train, y_train)
y_pred_hgb = model_hgb.predict(X_test)

mae_hgb = mean_absolute_error(y_test, y_pred_hgb)
mse_hgb = mean_squared_error(y_test, y_pred_hgb)
rmse_hgb = np.sqrt(mse_hgb)
r2_hgb = r2_score(y_test, y_pred_hgb)

print("Mean Absolute Error:", mae_hgb)
print("Mean Squared Error:", mse_hgb)
print("Root Mean Squared Error:", rmse_hgb)
print("R2-score:", r2_hgb)

Mean Absolute Error: 68348.03800426109
Mean Squared Error: 9995466615.046524
Root Mean Squared Error: 99977.33050570276
R2-score: 0.7867762639634341

# Model Comparison

In [1]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "SVR", "Decision Tree", "Random Forest", "HistGradientBoosting"],
    "R2 Score": [r2_lr, r2_svr, r2_dt, r2_rf, r2_hgb],
    "MAE ($)": [mae_lr, mae_svr, mae_dt, mae_rf, mae_hgb],
    "RMSE ($)": [rmse_lr, rmse_svr, rmse_dt, rmse_rf, rmse_hgb],
}).sort_values("R2 Score", ascending=False).reset_index(drop=True)

results

                  Model  R2 Score       MAE ($)       RMSE ($)
0  HistGradientBoosting  0.786776  68348.038004   99977.330506
1         Random Forest  0.756811  72331.121783  106771.591865
2                   SVR  0.742341  75314.787787  109902.123707
3     Linear Regression  0.737795  79446.106225  110867.561044
4         Decision Tree  0.679466  84261.674549  122580.264629
